# Deep Learning on MNIST: Perceptron, MLP and CNN

This project compares three neural-network approaches for classifying handwritten digits (0–9) from the MNIST dataset using PyTorch.

**Dataset:** MNIST (downloaded automatically through `torchvision`)  
**Models:** Linear classifier, multilayer perceptron (MLP), convolutional neural network (CNN)  
**Metrics:** Training loss and test accuracy

> The first run downloads MNIST. Training time depends on your hardware. Start with the small settings below, then increase epochs if needed.

## 1. Setup and reproducibility

In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
BATCH_SIZE = 128
EPOCHS = 3
LEARNING_RATE = 1e-3
DATA_DIR = "./data"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Download and inspect MNIST

In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print(f"Training examples: {len(train_dataset):,}")
print(f"Test examples: {len(test_dataset):,}")
print("Batch image shape:", tuple(images.shape))
print("Batch label shape:", tuple(labels.shape))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, image, label in zip(axes, images[:8], labels[:8]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(str(label.item()))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Define the models

In [ ]:
class LinearClassifier(nn.Module):
    """A single linear layer applied to flattened 28x28 images."""
    def __init__(self):
        super().__init__()
        self.classifier = nn.Linear(28 * 28, 10)

    def forward(self, x):
        return self.classifier(x.flatten(start_dim=1))


class MLP(nn.Module):
    """A fully connected network with two hidden layers."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.network(x)


class SimpleCNN(nn.Module):
    """A compact CNN that learns spatial features from image pixels."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


models = {
    "Linear classifier": LinearClassifier,
    "MLP": MLP,
    "CNN": SimpleCNN,
}

## 4. Training and evaluation helpers

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(inputs)
            loss = criterion(logits, targets)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * targets.size(0)
            correct += (logits.argmax(dim=1) == targets).sum().item()
            total += targets.size(0)

    return total_loss / total, 100.0 * correct / total


def train_model(model, train_loader, test_loader, epochs=EPOCHS, learning_rate=LEARNING_RATE):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "train_accuracy": [], "test_loss": [], "test_accuracy": []}

    for epoch in range(epochs):
        start = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        test_loss, test_acc = run_epoch(model, test_loader, criterion)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_accuracy"].append(test_acc)
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
            f"test loss {test_loss:.4f}, test acc {test_acc:.2f}% | "
            f"{time.time() - start:.1f}s"
        )
    return model, history

## 5. Train and compare models

In [ ]:
results = {}
trained_models = {}

for name, model_class in models.items():
    print(f"\n{'=' * 18} {name} {'=' * 18}")
    model, history = train_model(model_class(), train_loader, test_loader)
    trained_models[name] = model
    results[name] = history

## 6. Compare test accuracy

In [ ]:
model_names = list(results.keys())
final_accuracies = [results[name]["test_accuracy"][-1] for name in model_names]

plt.figure(figsize=(8, 4))
bars = plt.bar(model_names, final_accuracies)
plt.ylabel("Test accuracy (%)")
plt.title("MNIST model comparison")
plt.ylim(0, 100)
for bar, value in zip(bars, final_accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, value + 0.5, f"{value:.2f}%", ha="center")
plt.tight_layout()
plt.show()

for name, accuracy in zip(model_names, final_accuracies):
    print(f"{name}: {accuracy:.2f}%")

## 7. Plot loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in results.items():
    axes[0].plot(history["train_loss"], marker="o", label=name)
    axes[1].plot(history["test_accuracy"], marker="o", label=name)

axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].set_title("Test accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 8. Predict sample images

In [ ]:
best_name = max(results, key=lambda name: results[name]["test_accuracy"][-1])
best_model = trained_models[best_name]
best_model.eval()
sample_images, sample_labels = next(iter(test_loader))

with torch.no_grad():
    predictions = best_model(sample_images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for ax, image, actual, predicted in zip(axes.flat, sample_images[:10], sample_labels[:10], predictions[:10]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"Actual: {actual.item()}\nPred: {predicted.item()}")
    ax.axis("off")
plt.suptitle(f"Predictions from {best_name}")
plt.tight_layout()
plt.show()

## 9. Conclusions

- The linear classifier provides a simple baseline.
- The MLP can learn non-linear combinations of pixel features.
- The CNN uses local spatial patterns and is typically well suited to image classification.
- Results depend on the number of epochs, learning rate, random seed, and hardware. Record the results from your own run rather than assuming a fixed accuracy.

### Possible extensions
- Tune batch size, learning rate, optimizer, and hidden-layer sizes.
- Add a validation split for hyperparameter selection.
- Save the trained model and evaluate a confusion matrix.